# 00 - Setup Lakehouse

## Purpose

Define reusable parameters, paths, table names, and helper functions used by the remaining notebooks.

## Prerequisites

- A Fabric Lakehouse is attached as the default Lakehouse.
- CSV files are uploaded to Files/retail_banking/source/.
- You have permission to create Delta tables in the Lakehouse.

## Output

This notebook prints the run configuration and validates that the expected source paths are configured.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Environment parameters. In production, pass these from a Fabric Data Pipeline.
environment = "dev"
batch_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
source_base_path = "Files/retail_banking/source"

bronze_prefix = "bronze"
silver_prefix = "silver"
gold_prefix = "gold"

entities = ["customers", "accounts", "products", "branches", "transactions"]

print(f"Environment: {environment}")
print(f"Batch ID: {batch_id}")
print(f"Source base path: {source_base_path}")

In [ ]:
def source_file(entity_name: str) -> str:
    return f"{source_base_path}/{entity_name}.csv"


def bronze_table(entity_name: str) -> str:
    return f"{bronze_prefix}_{entity_name}"


def silver_table(entity_name: str) -> str:
    return f"{silver_prefix}_{entity_name}"


def write_delta_table(df, table_name: str, mode: str = "overwrite"):
    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Wrote table {table_name} with {df.count()} rows")


def log_count(label: str, df):
    count_value = df.count()
    print(f"{label}: {count_value} rows")
    return count_value

for entity in entities:
    print(f"Configured source for {entity}: {source_file(entity)}")

## Next Step

Run notebook 01 to ingest source CSV files into Bronze Delta tables.